# Notebook 3.1  Seeing speech: from waveform to features

**Companion to Chapter 3, *Introduction to Arabic Speech Technologies*.**

**Goal.** Compute and visualize every representation in the chapter on one signal: waveform,
narrowband vs wideband spectrograms, log-mel features, MFCCs, a SpecAugment view, and an LPC
estimate of the second formant for an emphatic vs plain syllable.

Self-contained (the signal is synthesized with numpy/scipy; no librosa or downloads needed).
The formal definitions are implemented directly below so you can inspect the computations.
Optional real audio: [Common Voice Arabic](https://commonvoice.mozilla.org/ar).

## 1. A synthetic Arabic vowel signal (16 kHz)

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, lfilter, get_window

sr = 16000; dur = 0.6
t = np.linspace(0, dur, int(sr*dur), endpoint=False)

def synth_vowel(f0, formants, sr, t):
    src = np.zeros_like(t); src[::int(sr/f0)] = 1.0
    sig = np.zeros_like(t)
    for fF, bw in formants:
        r = np.exp(-np.pi*bw/sr); th = 2*np.pi*fF/sr
        sig += lfilter([1], [1, -2*r*np.cos(th), r*r], src)
    return sig/np.max(np.abs(sig))

# a 'sawt'-like voiced signal
x = synth_vowel(120, [(600,80),(1200,90),(2500,120)], sr, t)
print('signal:', x.shape[0], 'samples =', dur, 's at', sr, 'Hz')

## 2. The waveform

In [ ]:
plt.figure(figsize=(10,2.6))
plt.plot(t[:1600], x[:1600], lw=0.8)
plt.xlabel('time (s)'); plt.ylabel('amplitude'); plt.title('Waveform (first 100 ms)')
plt.tight_layout(); plt.savefig('ch03_waveform.png', dpi=110); print('saved ch03_waveform.png')

## 3. Narrowband vs wideband spectrograms

A long window resolves the harmonics (pitch); a short window resolves the formants and timing.
This is the time-frequency trade-off.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4), sharey=True)
for a, (win, label) in zip(ax, [(1024,'narrowband (long window)'),(128,'wideband (short window)')]):
    f, tt, S = spectrogram(x, sr, nperseg=win, noverlap=int(win*0.75))
    a.pcolormesh(tt, f, 10*np.log10(S+1e-10), shading='gouraud')
    a.set_title(label); a.set_xlabel('time (s)'); a.set_ylim(0,4000)
ax[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.savefig('ch03_spectrograms.png', dpi=110); print('saved ch03_spectrograms.png')

## 4. Framing and the mel filterbank -> log-mel features

We frame the signal (25 ms / 10 ms), take the power spectrum, build ~40 triangular mel filters,
and take the logarithm. A 25 ms frame at 16 kHz is 400 samples; a 512-point FFT gives 257 bins.

In [ ]:
def frames(x, sr, win_ms=25, hop_ms=10):
    win = int(sr*win_ms/1000); hop = int(sr*hop_ms/1000)
    idx = range(0, len(x)-win, hop)
    w = get_window('hamming', win)
    return np.array([x[i:i+win]*w for i in idx]), win, hop

def hz_to_mel(f): return 2595*np.log10(1+f/700)
def mel_to_hz(m): return 700*(10**(m/2595)-1)

def mel_filterbank(n_filters, n_fft, sr, fmin=0, fmax=None):
    fmax = fmax or sr/2
    mels = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_filters+2)
    hz = mel_to_hz(mels)
    bins = np.floor((n_fft+1)*hz/sr).astype(int)
    fb = np.zeros((n_filters, n_fft//2+1))
    for m in range(1, n_filters+1):
        l,c,r = bins[m-1], bins[m], bins[m+1]
        for kk in range(l,c): fb[m-1,kk] = (kk-l)/max(c-l,1)
        for kk in range(c,r): fb[m-1,kk] = (r-kk)/max(r-c,1)
    return fb

n_fft = 512; n_mels = 40
F, win, hop = frames(x, sr)
mag = np.abs(np.fft.rfft(F, n=n_fft, axis=1))**2          # power spectrum: (frames, 257)
fb = mel_filterbank(n_mels, n_fft, sr)                    # (40, 257)
logmel = np.log(mag @ fb.T + 1e-10)                       # (frames, 40)
print('frame len =', win, 'samples; power bins =', mag.shape[1], '; log-mel =', logmel.shape)

plt.figure(figsize=(10,3.2))
plt.imshow(logmel.T, origin='lower', aspect='auto', cmap='magma')
plt.xlabel('frame'); plt.ylabel('mel filter'); plt.title('Log-mel features (40 filters)')
plt.colorbar(label='log energy'); plt.tight_layout()
plt.savefig('ch03_logmel.png', dpi=110); print('saved ch03_logmel.png')

## 5. MFCCs (DCT of the log-mel features)

In [ ]:
from scipy.fftpack import dct
mfcc = dct(logmel, type=2, axis=1, norm='ortho')[:, :13]   # keep 13 coefficients
print('MFCC shape:', mfcc.shape, '(frames, 13)')
plt.figure(figsize=(10,3))
plt.imshow(mfcc.T, origin='lower', aspect='auto', cmap='viridis')
plt.xlabel('frame'); plt.ylabel('MFCC'); plt.title('MFCCs (13 coefficients)')
plt.colorbar(); plt.tight_layout(); plt.savefig('ch03_mfcc.png', dpi=110); print('saved ch03_mfcc.png')

## 6. SpecAugment (training-time only)

Mask a band of frequency and a span of time in the log-mel features. Applied to training data
only; never at test time.

In [ ]:
rng = np.random.default_rng(0)
aug = logmel.copy()
f0 = rng.integers(0, n_mels-8); aug[:, f0:f0+8] = aug.min()        # frequency mask
t0 = rng.integers(0, max(1, aug.shape[0]-15)); aug[t0:t0+15, :] = aug.min()  # time mask
plt.figure(figsize=(10,3.2))
plt.imshow(aug.T, origin='lower', aspect='auto', cmap='magma')
plt.xlabel('frame'); plt.ylabel('mel filter'); plt.title('SpecAugment: frequency + time masks')
plt.tight_layout(); plt.savefig('ch03_specaugment.png', dpi=110); print('saved ch03_specaugment.png')

## 7. LPC: estimate F2 for an emphatic vs a plain vowel

Linear prediction fits an all-pole filter whose peaks are the formants. We synthesize a plain
vowel (high F2) and an emphatic-context vowel (low F2) and recover F2 from the LPC spectrum.

In [ ]:
def lpc(sig, order):
    # autocorrelation + Levinson-Durbin
    sig = sig*get_window('hamming', len(sig))
    r = np.correlate(sig, sig, 'full')[len(sig)-1:len(sig)+order]
    a = np.zeros(order+1); a[0] = 1.0; e = r[0]
    if e == 0: return a
    for i in range(1, order+1):
        k = -(a[1:i] @ r[i-1:0:-1] + r[i]) / e if i>1 else -r[1]/r[0]
        a_new = a.copy()
        for j in range(1, i): a_new[j] = a[j] + k*a[i-j]
        a_new[i] = k; a = a_new; e *= (1-k*k)
        if e <= 0: break
    return a

def formants_from_lpc(sig, sr, order=12):
    a = lpc(sig, order)
    roots = np.roots(a)
    roots = roots[np.imag(roots) >= 0.01]
    angs = np.arctan2(np.imag(roots), np.real(roots))
    freqs = sorted(angs*sr/(2*np.pi))
    return [f for f in freqs if 90 < f < 4000]

plain    = synth_vowel(120, [(300,60),(2300,90),(3000,120)], sr, t)
emphatic = synth_vowel(120, [(350,60),(1600,90),(2700,120)], sr, t)
seg = slice(int(0.2*sr), int(0.2*sr)+400)  # mid-vowel, 25 ms
print('plain    formants (Hz):', [round(f) for f in formants_from_lpc(plain[seg], sr)])
print('emphatic formants (Hz):', [round(f) for f in formants_from_lpc(emphatic[seg], sr)])
print('=> the second formant is lower in the emphatic context, as Section 3.7 describes.')

## 8. Exercise solutions

Runnable code for the chapter exercises. Exercises 1-3 are programmatic and solved in full;
Exercises 4 and 5 are design questions, summarized here. These cells reuse functions defined
earlier in this notebook (`synth_vowel`, `formants_from_lpc`, `frames`, `mel_filterbank`).

**Exercise 1.** Generate a 9 kHz tone, sample at 16 kHz with no anti-aliasing filter, and find the aliased tone.

In [ ]:
sr = 16000
t1 = np.linspace(0, 0.25, int(sr*0.25), endpoint=False)
tone = np.sin(2*np.pi*9000*t1)            # a 9 kHz tone, already 'sampled' at 16 kHz
spec = np.abs(np.fft.rfft(tone*np.hanning(len(tone))))
freqs = np.fft.rfftfreq(len(tone), 1/sr)
peak = freqs[np.argmax(spec)]
print(f'observed peak: {peak:.0f} Hz  (9 kHz aliases to 16000 - 9000 = 7000 Hz)')
print('min rate for content up to 7 kHz: > 14 kHz; choose 16 kHz; cutoff below the 8 kHz Nyquist limit.')

**Exercise 2.** Measure F2 for an emphatic vs a plain vowel and compare. With a corpus you would extract many tokens; here we synthesize several noisy instances of each.

In [ ]:
rng = np.random.default_rng(0)
def f2_sample(f2_center):
    v = synth_vowel(120, [(320,60),(f2_center+rng.normal(0,60),90),(2900,120)], sr, t)
    fs = formants_from_lpc(v[int(0.2*sr):int(0.2*sr)+400], sr)
    return min(fs[1:], key=lambda x: abs(x-f2_center)) if len(fs) > 1 else fs[0]
plain_f2    = [f2_sample(2300) for _ in range(10)]   # plain /t/ context
emphatic_f2 = [f2_sample(1600) for _ in range(10)]   # emphatic /tQ/ context
print('plain    F2 median: {:.0f} Hz'.format(np.median(plain_f2)))
print('emphatic F2 median: {:.0f} Hz'.format(np.median(emphatic_f2)))
print('=> the emphatic distribution sits lower; use the wideband view to locate the vowel.')

**Exercise 3.** Build a log-mel pipeline, print the size after each stage, and note the toolkit comparison.

In [ ]:
x3 = synth_vowel(120, [(600,80),(1200,90),(2500,120)], sr, t)
F3, win3, hop3 = frames(x3, sr)
mag3 = np.abs(np.fft.rfft(F3, n=512, axis=1))**2
fb3 = mel_filterbank(40, 512, sr)
logmel3 = np.log(mag3 @ fb3.T + 1e-10)
from scipy.fftpack import dct as _dct
mfcc3 = _dct(logmel3, type=2, axis=1, norm='ortho')[:, :13]
print('frame samples:', win3, '-> power bins:', mag3.shape[1], '-> mel:', fb3.shape[0],
      '-> log-mel:', logmel3.shape, '-> MFCC:', mfcc3.shape)
print('a standard toolkit (e.g. librosa) matches this up to mel-formula, power-vs-magnitude,')
print('and log-floor conventions, which is why those settings must be reported.')

**Exercise 4 (design).** A 16 kHz model fails on 8 kHz audio because the 8 kHz audio has no energy above 4 kHz (empty upper mel bands) and telephone handsets add channel coloring. Fix: work at 8 kHz with mel filters over 0-4 kHz, use per-speaker or per-call-side CMVN, and fine-tune on telephone-band data.

**Exercise 5 (design).** Split by speaker (for example 40 train / 5 dev / 5 test), with no speaker in more than one set and dialect and gender balanced. The protecting rule is no speaker overlap; splitting by utterance leaks voices into the test set and inflates the score on unseen users.